# 01. 입력 검증과 준비

**목적:** 최종본의 고정 매니페스트와 입력 해시를 확인하고 종로구·마포구의 상권 격자와 공연 연결 검증용 두 업종 패널을 만든다. 기본 연구 범위는 **외식업 10종, 2023~2025년 12개 분기**이며, 두 업종은 연결·기준선 검증을 위한 중간 자료다. 최종 10종 패널은 04에서 만든다.

`final/metadata/`의 검증된 입력만 오프라인으로 읽는다. 원자료는 수정하지 않으며 파생 자료는 `data/analysis/final/`, 보고서는 `reports/final/`에 저장한다. 매출 미제공을 0원으로 바꾸지 않는다.

In [1]:
from pathlib import Path
import json
import hashlib
import sys
import platform
import importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# 프로젝트 루트 또는 notebooks 폴더에서 실행할 수 있습니다.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/raw/seoul_sales').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('졸업프로젝트 폴더 또는 notebooks 폴더에서 실행하세요.')
RAW = ROOT / 'data/raw'
ANALYSIS = ROOT / 'data/analysis/final'
REPORTS = ROOT / 'reports/final'
FIGURES = REPORTS / 'figures'
for folder in [ANALYSIS, REPORTS, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 16)
pd.set_option('display.max_rows', 16)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')
YEARS = [2023, 2024, 2025]
DISTRICTS = {'11110': '종로구', '11440': '마포구'}
INDUSTRIES = {'CS100001': '한식음식점', 'CS100010': '커피-음료'}
KEY = ['quarter', 'area_id', 'industry_id']
print('프로젝트:', ROOT)
print('Python:', platform.python_version())


프로젝트: C:\Coding\졸업프로젝트
Python: 3.12.14


## 1. 고정 입력과 해시 검증

서울시 매출·점포·경계와 KOPIS 목록·상세의 경로를 복사된 매니페스트에서 읽고 SHA-256을 확인한다. 실행 시점의 수집 중 CSV나 최신 포인터를 따라가지 않는다. 서울시 열 이름에 ‘당월’이 있어도 분석 단위는 분기다.

In [2]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

seoul_manifest = json.loads((ROOT / 'final/metadata/seoul_analysis_sources.json').read_text(encoding='utf-8'))
kopis_manifest = json.loads((ROOT / 'final/metadata/kopis_analysis_sources.json').read_text(encoding='utf-8'))
assert seoul_manifest['status'] == 'validated' and seoul_manifest['validation_passed']
assert kopis_manifest['validated'] is True
source_csv_paths, input_records = {}, []
for item in seoul_manifest['items']:
    for record in item['extracted']:
        path = ROOT / record['path']
        assert sha256_file(path) == record['sha256'], path.name
        input_records.append({'role': item['dataset'], 'path': record['path'], 'sha256': record['sha256']})
    if item['dataset'] in ['sales', 'stores']:
        csvs = [record for record in item['extracted'] if record['path'].endswith('.csv')]
        assert len(csvs) == 1
        source_csv_paths[(item['dataset'], int(item['year']))] = ROOT / csvs[0]['path']
    elif item['dataset'] == 'areas':
        boundary_path = ROOT / next(r['path'] for r in item['extracted'] if r['path'].endswith('.shp'))
assert set(source_csv_paths) == {(kind, year) for kind in ['sales', 'stores'] for year in YEARS}
for role, record in [(f'kopis_input_{i}', r) for i, r in enumerate(kopis_manifest['input_sources'])] + list(kopis_manifest['files'].items()):
    assert sha256_file(ROOT / record['path']) == record['sha256'], record['path']
    input_records.append({'role': role, 'path': record['path'], 'sha256': record['sha256']})
validation_sources = json.loads((ROOT / 'final/metadata/validation_sources.json').read_text(encoding='utf-8'))
for record in validation_sources['files']:
    assert sha256_file(ROOT / record['path']) == record['sha256'], record['path']
    input_records.append(record)
input_integrity = pd.DataFrame(input_records)
input_integrity.to_csv(REPORTS / '01_fixed_input_fingerprints.csv', index=False, encoding='utf-8-sig')
display(input_integrity[['role', 'path']])


,role,path
0,sales,data/raw/seoul_recollection/20261004_091155_b8...
1,sales,data/raw/seoul_recollection/20261004_091155_b8...
2,sales,data/raw/seoul_recollection/20261004_091155_b8...
3,stores,data/raw/seoul_recollection/20261004_091155_b8...
4,stores,data/raw/seoul_recollection/20261004_091155_b8...
...,...,...
398,monthly_xml,data/raw/kopis/performance_lists/202512/014.xml
399,monthly_xml,data/raw/kopis/performance_lists/202512/015.xml
400,baseline,data/analysis/baseline_two_industry/sales_stor...
401,baseline,data/analysis/baseline_two_industry/performanc...


**해석:** 해시 일치는 지정된 입력의 재현성을 뜻한다. 자료의 모집단 완전성이나 과거 경계·시설 위치의 정확성을 보증하지 않는다.

## 2. 원본 구조·결측·중복 점검

분기×상권×업종 키가 중복되면 중단한다. 점포 원자료의 `점포_수`는 **일반점포**, `프랜차이즈_점포_수`는 **프랜차이즈점포**, `유사_업종_점포_수`는 두 종류를 합친 **전체점포**다. `store_count`는 기준선 비교용 일반점포 별칭으로 유지한다.

In [3]:
SALES_NAMES = {
    '기준_년분기_코드': 'quarter', '상권_코드': 'area_id',
    '서비스_업종_코드': 'industry_id', '당월_매출_금액': 'total_sales',
    '당월_매출_건수': 'sales_transactions', '시간대_21~24_매출_금액': 'night_sales',
    '주말_매출_금액': 'weekend_sales', '주중_매출_금액': 'weekday_sales',
}
STORE_NAMES = {
    '기준_년분기_코드': 'quarter', '상권_코드': 'area_id', '서비스_업종_코드': 'industry_id',
    '점포_수': 'store_count', '유사_업종_점포_수': 'similar_industry_store_count',
    '개업_점포_수': 'opened_stores', '폐업_점포_수': 'closed_stores',
    '프랜차이즈_점포_수': 'franchise_stores',
    'stdr_yyqu_cd': 'quarter', 'trdar_cd': 'area_id', 'svc_induty_cd': 'industry_id',
    'stor_co': 'store_count', 'similr_induty_stor_co': 'similar_industry_store_count',
    'opbiz_stor_co': 'opened_stores', 'clsbiz_stor_co': 'closed_stores',
    'frc_stor_co': 'franchise_stores',
}
quality_rows, missing_rows = [], []
sales_frames, store_frames = [], []

for year in YEARS:
    for kind, rename, frames in [('sales', SALES_NAMES, sales_frames),
                                  ('stores', STORE_NAMES, store_frames)]:
        path = source_csv_paths[(kind, year)]
        raw = pd.read_csv(path, encoding='cp949', low_memory=False)
        for column in raw.columns:
            missing_rows.append({'dataset': kind, 'year': year, 'column': column,
                                 'missing_n': int(raw[column].isna().sum()),
                                 'missing_pct': float(raw[column].isna().mean() * 100)})
        frame = raw.rename(columns=rename)
        needed = list(dict.fromkeys(rename.values()))
        assert set(needed).issubset(frame.columns), f'{path.name}: 필수 열이 없습니다.'
        frame = frame[needed].copy()
        for column in KEY:
            frame[column] = frame[column].astype('string').str.replace(r'\.0$', '', regex=True)
        duplicates = int(frame.duplicated(KEY).sum())
        quality_rows.append({'dataset': kind, 'year': year, 'rows': len(frame),
                             'columns_in_raw': raw.shape[1], 'duplicate_keys': duplicates,
                             'missing_key_rows': int(frame[KEY].isna().any(axis=1).sum())})
        assert duplicates == 0 and not frame[KEY].isna().any().any(), path.name
        assert set(frame['quarter']) == {f'{year}{q}' for q in range(1, 5)}, path.name
        frames.append(frame)

sales = pd.concat(sales_frames, ignore_index=True)
stores = pd.concat(store_frames, ignore_index=True)
quality = pd.DataFrame(quality_rows)
missing_source = pd.DataFrame(missing_rows)
quality.to_csv(REPORTS / 'source_quality.csv', index=False, encoding='utf-8-sig')
missing_source.to_csv(REPORTS / 'source_column_missingness.csv', index=False, encoding='utf-8-sig')
display(quality)
display(missing_source.query('missing_n > 0').head(12))


,dataset,year,rows,columns_in_raw,duplicate_keys,missing_key_rows
0,sales,2023,88246,55,0,0
1,stores,2023,307741,14,0,0
2,sales,2024,87179,55,0,0
3,stores,2024,306889,14,0,0
4,sales,2025,85732,55,0,0
5,stores,2025,304775,14,0,0


,dataset,year,column,missing_n,missing_pct


## 3. 연구 지역과 업종 선택

상권 경계 파일의 자치구 코드를 이용해 종로구와 마포구를 선택합니다. **현재 확보한 경계 스냅샷**을 2023~2025년에 공통으로 적용합니다. 과거 경계가 같았다는 뜻은 아니며, 경계 변경 검증은 별도 과제입니다.

2022년 시범 다운로드는 `data/raw/legacy_downloads/`에 보존하지만 본 분석에는 포함하지 않습니다.


In [4]:
import shapefile

boundary_reader = shapefile.Reader(str(boundary_path),
    encoding=boundary_path.with_suffix('.cpg').read_text().strip())
area_attributes = pd.DataFrame([record.as_dict() for record in boundary_reader.iterRecords()])
areas = area_attributes.rename(columns={'TRDAR_CD': 'area_id', 'TRDAR_CD_N': 'area_name',
                                        'SIGNGU_CD': 'district_code'})
for column in ['area_id', 'district_code']:
    areas[column] = areas[column].astype('string').str.replace(r'\.0$', '', regex=True)
areas = areas.loc[areas['district_code'].isin(DISTRICTS),
                  ['area_id', 'area_name', 'district_code']].copy()
areas['district'] = areas['district_code'].map(DISTRICTS)
assert not areas['area_id'].duplicated().any()
areas.to_csv(ANALYSIS / 'study_areas.csv', index=False, encoding='utf-8-sig')

def select_study(frame):
    return frame.loc[frame['area_id'].isin(areas['area_id']) &
                     frame['industry_id'].isin(INDUSTRIES)].copy()

sales_study = select_study(sales)
stores_study = select_study(stores)
display(areas.groupby('district', observed=True).size().rename('상권 수').to_frame())
print('관측된 매출 행:', len(sales_study), '/ 관측된 점포 행:', len(stores_study))


,상권 수
district,
마포구,77
종로구,74


관측된 매출 행: 2797 / 관측된 점포 행: 3367


## 4. 결측을 보존한 패널 만들기

상권×2개 업종×12개 분기의 틀에 매출과 점포를 각각 연결합니다. `sales_record_present`는 원자료에 그 관측이 있었는지를 뜻합니다. 점포 자료가 있는데 매출 자료가 없는 경우에도 매출을 0으로 채우지 않습니다.

두 원자료가 모두 없는 행은 영업 부재인지 공개자료 누락인지 판단할 수 없어 `both_absent`로 남깁니다.


In [5]:
quarters = pd.DataFrame({'quarter': [f'{year}{q}' for year in YEARS for q in range(1, 5)]})
industries = pd.DataFrame({'industry_id': list(INDUSTRIES), 'industry': list(INDUSTRIES.values())})
panel = areas.merge(industries, how='cross').merge(quarters, how='cross')
sales_study['sales_record_present'] = True
stores_study['store_record_present'] = True
panel = panel.merge(sales_study, on=KEY, how='left', validate='one_to_one')
panel = panel.merge(stores_study, on=KEY, how='left', validate='one_to_one')
for column in ['sales_record_present', 'store_record_present']:
    panel[column] = panel[column].eq(True)
panel['availability'] = np.select(
    [panel['sales_record_present'] & panel['store_record_present'],
     ~panel['sales_record_present'] & panel['store_record_present'],
     panel['sales_record_present'] & ~panel['store_record_present']],
    ['both_present', 'sales_absent', 'stores_absent'], default='both_absent')
assert len(panel) == len(areas) * len(INDUSTRIES) * len(quarters)
assert not panel.duplicated(KEY).any()
assert panel.loc[~panel['sales_record_present'], 'total_sales'].isna().all()
display(panel.groupby(['district', 'industry', 'availability'], observed=True)
        .size().rename('행 수').reset_index())


,district,industry,availability,행 수
0,마포구,커피-음료,both_absent,25
1,마포구,커피-음료,both_present,708
2,마포구,커피-음료,sales_absent,191
3,마포구,한식음식점,both_absent,21
4,마포구,한식음식점,both_present,777
5,마포구,한식음식점,sales_absent,126
6,종로구,커피-음료,both_absent,104
7,종로구,커피-음료,both_present,600
8,종로구,커피-음료,sales_absent,184
9,종로구,한식음식점,both_absent,107


## 5. 금액과 비율의 유효성 확인

야간 매출 비중 = 21~24시 매출 ÷ 총매출, 주말 매출 비중 = 주말 매출 ÷ 총매출입니다. **주말 야간 매출**은 이 두 지표만으로 계산하지 않습니다.

분모가 0이거나 음수 금액, 부분 매출이 총매출보다 큰 경우 비율은 결측으로 둡니다. 원본 금액은 보존하고 이상 여부만 표시합니다. 합계 차이는 추정·반올림 영향도 있을 수 있어 삭제 사유로 바로 사용하지 않습니다.

매출 추정 모집단과 점포 모집단의 일치를 확인하지 않았으므로 점포당 매출은 만들지 않는다.

In [6]:
# 원자료 항등식: 전체 = 일반 + 프랜차이즈. 기존 이름은 기준선 비교용 별칭입니다.
panel['regular_store_count'] = panel['store_count']
panel['franchise_store_count'] = panel['franchise_stores']
panel['total_store_count'] = panel['similar_industry_store_count']
observed_stores = panel['store_record_present']
assert panel.loc[observed_stores, 'total_store_count'].eq(
    panel.loc[observed_stores, 'regular_store_count'] + panel.loc[observed_stores, 'franchise_store_count']).all()
panel['negative_sales_flag'] = panel[['total_sales', 'night_sales', 'weekend_sales',
                                         'weekday_sales']].lt(0).any(axis=1)
panel['negative_store_flag'] = panel[['regular_store_count', 'franchise_store_count', 'total_store_count']].lt(0).any(axis=1)
panel['night_exceeds_total'] = panel['night_sales'].gt(panel['total_sales'])
panel['weekend_exceeds_total'] = panel['weekend_sales'].gt(panel['total_sales'])
panel['weekday_weekend_difference'] = panel['weekday_sales'] + panel['weekend_sales'] - panel['total_sales']
for component, output in [('night_sales', 'night_share'), ('weekend_sales', 'weekend_share')]:
    valid = panel['total_sales'].gt(0) & panel[component].ge(0) & panel[component].le(panel['total_sales'])
    panel[output] = (panel[component] / panel['total_sales']).where(valid)
panel['year'] = panel['quarter'].str[:4].astype(int)
panel['quarter_number'] = panel['quarter'].str[-1].astype(int)
flag_columns = ['negative_sales_flag', 'negative_store_flag', 'night_exceeds_total', 'weekend_exceeds_total']
flag_summary = panel[flag_columns].sum().rename('flagged_rows').to_frame()
missing_panel = panel[['total_sales', 'store_count', 'night_share', 'weekend_share']]
missing_panel = missing_panel.isna().agg(['sum', 'mean']).T.rename(columns={'sum': 'missing_n', 'mean': 'missing_fraction'})
display(flag_summary)
display(missing_panel)
print('원자료에 제공된 총매출 0원 행:', int(panel['total_sales'].eq(0).sum()))
flag_summary.to_csv(REPORTS / 'value_flags.csv', encoding='utf-8-sig')
missing_panel.to_csv(REPORTS / 'panel_missingness.csv', encoding='utf-8-sig')
panel.loc[panel[flag_columns].any(axis=1)].to_csv(REPORTS / 'flagged_panel_rows.csv', index=False, encoding='utf-8-sig')


,flagged_rows
negative_sales_flag,0
negative_store_flag,0
night_exceeds_total,0
weekend_exceeds_total,0


,missing_n,missing_fraction
total_sales,827.000,0.228
store_count,257.000,0.071
night_share,827.000,0.228
weekend_share,827.000,0.228


원자료에 제공된 총매출 0원 행: 0


## 6. 분석용 자료 저장

다음 노트북은 이 패널을 읽습니다. 원자료의 미제공 매출은 그대로 결측이며, 비율을 정의할 수 없는 행도 자동 보간하지 않았습니다. 관측 가능한 표본의 규모를 확인한 뒤 분석 목적별로 사용할 행을 선택합니다.


In [7]:
panel = panel.sort_values(KEY).reset_index(drop=True)
panel.to_csv(ANALYSIS / 'sales_stores_panel.csv', index=False, encoding='utf-8-sig')
coverage = panel.groupby(['district', 'industry', 'quarter'], observed=True).agg(
    expected_rows=('area_id', 'size'), sales_rows=('sales_record_present', 'sum'),
    store_rows=('store_record_present', 'sum'))
coverage['sales_coverage_pct'] = coverage['sales_rows'] / coverage['expected_rows'] * 100
coverage.to_csv(REPORTS / 'panel_coverage.csv', encoding='utf-8-sig')
packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'statsmodels', 'scipy',
            'pyshp', 'pyproj', 'shapely', 'nbformat', 'nbclient', 'ipykernel', 'jupyterlab']
versions = {name: importlib.metadata.version(name) for name in packages}
(REPORTS / 'python_environment.json').write_text(json.dumps(
    {'python': platform.python_version(), 'packages': versions}, ensure_ascii=False, indent=2), encoding='utf-8')
summary = {'areas': len(areas), 'panel_rows': len(panel), 'observed_sales_rows': int(panel['sales_record_present'].sum()),
           'sales_absent_rows': int((~panel['sales_record_present']).sum()), 'years': YEARS,
           'districts': list(DISTRICTS.values()), 'industries': list(INDUSTRIES.values())}
(REPORTS / 'preprocessing_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame([summary]))
display(panel[KEY + ['district', 'industry', 'total_sales', 'night_share', 'store_count', 'availability']].head(8))
print('저장:', ANALYSIS / 'sales_stores_panel.csv')

assert all(sha256_file(ROOT / r.path) == r.sha256 for r in input_integrity.itertuples())
print('고정 입력 실행 전후 해시 일치:', len(input_integrity), '개')


,areas,panel_rows,observed_sales_rows,sales_absent_rows,years,districts,industries
0,151,3624,2797,827,"[2023, 2024, 2025]","[종로구, 마포구]","[한식음식점, 커피-음료]"


,quarter,area_id,industry_id,district,industry,total_sales,night_share,store_count,availability
0,20231,3001494,CS100001,종로구,한식음식점,"46,431,279,586.000",0.159,440.000,both_present
1,20231,3001494,CS100010,종로구,커피-음료,"6,876,548,548.000",0.047,112.000,both_present
2,20231,3110001,CS100001,종로구,한식음식점,"395,528,121.000",0.024,10.000,both_present
3,20231,3110001,CS100010,종로구,커피-음료,NaN,NaN,1.000,sales_absent
4,20231,3110002,CS100001,종로구,한식음식점,"524,733,990.000",0.029,8.000,both_present
5,20231,3110002,CS100010,종로구,커피-음료,"75,049,747.000",0.000,2.000,both_present
6,20231,3110003,CS100001,종로구,한식음식점,"224,822,221.000",0.060,6.000,both_present
7,20231,3110003,CS100010,종로구,커피-음료,"35,953,346.000",0.000,5.000,both_present


저장: C:\Coding\졸업프로젝트\data\analysis\final\sales_stores_panel.csv


고정 입력 실행 전후 해시 일치: 403 개


**해석:** 이 패널의 두 업종은 연결 검증용이다. 02에서 공연·시설을 연결하고 03에서 재계산한 뒤, 04에서 외식업 10종 전체의 분석 패널을 만든다.